# Ejercicio 2 — Modelo de Machine Learning de clasificación

## Versión resuelta orientativa

## Bootcamp ANBAN · Machine Learning básico

En este notebook vas a construir un flujo completo de **Machine Learning supervisado para clasificación**. La estructura será muy parecida a la del ejercicio de regresión, pero ahora la variable objetivo no será un número continuo, sino una **categoría**.

El objetivo es practicar el flujo completo:

1. cargar un dataset público,
2. entender qué clase queremos predecir,
3. hacer una preparación mínima del dato,
4. separar entrenamiento y test correctamente,
5. construir transformadores y modelos con `scikit-learn`,
6. comparar varios clasificadores,
7. interpretar métricas de clasificación.

> **Importante:** este notebook no contiene código resuelto. Cada celda de código vacía está pensada para que implementes tú el paso indicado justo encima.


## Dataset propuesto

Trabajaremos con el dataset público **Palmer Penguins**.

La tarea consiste en predecir la especie de un pingüino a partir de medidas corporales y variables descriptivas.

**URL de lectura del CSV:**

`https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv`

La variable objetivo será:

- `species`: especie del pingüino.

Variables disponibles esperadas:

- `species`: especie del pingüino. Será la variable objetivo.
- `island`: isla donde se observó el pingüino.
- `bill_length_mm`: longitud del pico.
- `bill_depth_mm`: profundidad del pico.
- `flipper_length_mm`: longitud de la aleta.
- `body_mass_g`: masa corporal.
- `sex`: sexo del pingüino.

Este dataset es pequeño, visual y muy adecuado para practicar clasificación multiclase con variables numéricas y categóricas.


## Normas del ejercicio

- No uses el dataset trabajado en el notebook de clase.
- Mantén el nombre del dataframe principal como `df`.
- Mantén la variable objetivo con el nombre `y`.
- Mantén la matriz de variables predictoras con el nombre `X`.
- Usa una separación de **80% entrenamiento** y **20% test**.
- Usa `random_state=42` cuando haya aleatoriedad.
- Usa separación estratificada, porque queremos que las especies queden representadas de forma parecida en train y test.
- Evalúa todos los modelos sobre el mismo conjunto de test.
- No ajustes transformadores con todo el dataset antes de separar train/test. Recuerda el riesgo de *data leakage*.


## Modelos mínimos a comparar

Como mínimo, debes entrenar y evaluar estos modelos:

1. Logistic Regression,
2. KNN Classifier,
3. Support Vector Classifier,
4. Decision Tree Classifier,
5. Random Forest Classifier.

También se recomienda incluir un modelo baseline muy sencillo, por ejemplo un clasificador que siempre prediga la clase más frecuente.


## Celda 1 — Importación de librerías

Importa las librerías necesarias para trabajar con datos, visualizar información y construir modelos con `scikit-learn`.

Deberías importar herramientas para:

- manejar dataframes,
- hacer operaciones numéricas,
- crear gráficos básicos,
- separar train/test,
- construir pipelines,
- transformar variables numéricas y categóricas,
- entrenar los clasificadores pedidos,
- calcular métricas de clasificación.

No hace falta importar todo `sklearn` entero. Importa solo lo que vayas a usar.

**Qué deberías comprobar al terminar esta celda**

La celda debe ejecutarse sin errores. Todavía no debería cargar datos ni entrenar modelos.


In [ ]:
# Manipulación de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Separación de datos y validación
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Preprocesado
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Modelos de clasificación
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Configuración visual
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## Celda 2 — Carga del dataset

Carga el CSV desde la URL indicada y guárdalo en una variable llamada `df`.

Después muestra:

- las primeras filas,
- el tamaño del dataframe,
- los nombres de las columnas.

La finalidad de esta celda es comprobar que los datos han llegado correctamente antes de hacer limpieza o modelado.

**Qué deberías comprobar al terminar esta celda**

Deberías ver columnas como `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` y `sex`. El dataset debería tener algo más de 300 filas.


In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url)

print("Primeras filas del dataset:")
display(df.head())

print("\nTamaño del dataframe:")
print(df.shape)

print("\nColumnas disponibles:")
print(df.columns.tolist())

## Celda 3 — Primera inspección del dataset

Explora la estructura del dataframe.

En esta celda debes comprobar:

- el tipo de dato de cada columna,
- si hay valores nulos,
- si hay filas duplicadas,
- un resumen estadístico de las columnas numéricas,
- un resumen básico de las columnas categóricas.

No limpies todavía. Primero observa el problema.

**Qué deberías comprobar al terminar esta celda**

Deberías detectar que hay variables numéricas y categóricas. También es probable que aparezcan algunos valores nulos en columnas concretas.


In [ ]:
print("Información general del dataframe:")
df.info()

print("\nValores nulos por columna:")
display(df.isna().sum())

print("\nFilas duplicadas:")
print(df.duplicated().sum())

print("\nResumen estadístico de variables numéricas:")
display(df.describe())

print("\nResumen de variables categóricas:")
display(df.describe(include="object"))

## Celda 4 — Definición del problema de clasificación

Confirma que la variable objetivo será `species`.

En esta celda debes:

- comprobar cuántas especies distintas hay,
- contar cuántas observaciones hay de cada especie,
- interpretar si el problema es binario o multiclase,
- observar si las clases están muy desbalanceadas o razonablemente equilibradas.

**Qué deberías comprobar al terminar esta celda**

Deberías ver varias clases de pingüino. Este será un problema de clasificación multiclase.


In [ ]:
target = "species"

print("Variable objetivo:", target)
print("Tipo de problema: clasificación supervisada multiclase")
print("Queremos predecir la especie de cada pingüino.")

print("\nClases disponibles y número de ejemplos:")
display(df[target].value_counts())

## Celda 5 — Limpieza mínima del dato

Crea una versión limpia del dataframe.

Para este ejercicio, usa solo las columnas relevantes:

- `species`,
- `island`,
- `bill_length_mm`,
- `bill_depth_mm`,
- `flipper_length_mm`,
- `body_mass_g`,
- `sex`.

Después elimina o trata los valores nulos de esas columnas. Como el dataset es pequeño pero los nulos no deberían ser muchísimos, puedes eliminar las filas incompletas para simplificar.

También revisa si hay duplicados.

**Qué deberías comprobar al terminar esta celda**

Al terminar, deberías tener un dataframe sin nulos en las columnas que vas a usar para modelar.


In [ ]:
df_clean = df.copy()

# Eliminamos duplicados si existieran.
df_clean = df_clean.drop_duplicates()

# Eliminamos filas con valores nulos.
# En este dataset suele haber algunos nulos en medidas corporales y en sex.
df_clean = df_clean.dropna()

print("Tamaño original:", df.shape)
print("Tamaño tras limpieza mínima:", df_clean.shape)

print("\nValores nulos tras limpieza:")
display(df_clean.isna().sum())

print("\nDistribución de clases tras limpieza:")
display(df_clean[target].value_counts())

## Celda 6 — Exploración visual mínima

Realiza una exploración visual breve para entender el problema.

Como mínimo, intenta observar:

- distribución de especies,
- relación entre longitud y profundidad del pico según especie,
- relación entre masa corporal y especie,
- si la isla o el sexo parecen aportar información.

Después añade una breve reflexión en Markdown.

**Qué deberías comprobar al terminar esta celda**

Deberías observar que algunas variables separan bastante bien algunas especies, aunque puede haber zonas de solapamiento.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df_clean, x="species", ax=axes[0])
axes[0].set_title("Número de ejemplos por especie")

sns.scatterplot(
    data=df_clean,
    x="bill_length_mm",
    y="bill_depth_mm",
    hue="species",
    style="species",
    ax=axes[1]
)
axes[1].set_title("Relación entre largo y profundidad del pico")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x="species", y="flipper_length_mm", ax=axes[0])
axes[0].set_title("Longitud de aleta por especie")

sns.boxplot(data=df_clean, x="species", y="body_mass_g", ax=axes[1])
axes[1].set_title("Masa corporal por especie")

plt.tight_layout()
plt.show()

### Observaciones de la exploración

Solución orientativa:

- El dataset tiene tres clases: `Adelie`, `Chinstrap` y `Gentoo`.
- Hay algunos valores nulos, especialmente en variables corporales y en la variable `sex`; por simplicidad los eliminamos.
- Las especies no están exactamente igual de representadas, por lo que conviene usar `stratify=y` al hacer el `train_test_split`.
- Algunas variables numéricas, como `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` y `body_mass_g`, muestran diferencias claras entre especies.
- Las variables categóricas, como `island` y `sex`, necesitan codificarse antes de entrar al modelo.

## Celda 7 — Separación entre variables predictoras y variable objetivo

Crea las variables `X` e `y`.

- `y` debe contener únicamente la columna `species`.
- `X` debe contener las variables que usarás para predecir la especie.

No incluyas `species` dentro de `X`. Eso sería una fuga de información.

Comprueba las dimensiones de `X` e `y` al terminar.

**Qué deberías comprobar al terminar esta celda**

`X` debería contener las columnas predictoras y `y` debería contener la especie. Ambas estructuras deben tener el mismo número de filas.


In [ ]:
X = df_clean.drop(columns=target)
y = df_clean[target]

print("Tamaño de X:", X.shape)
print("Tamaño de y:", y.shape)

print("\nPrimeras filas de X:")
display(X.head())

print("\nPrimeros valores de y:")
display(y.head())

## Celda 8 — Identificación de columnas numéricas y categóricas

Crea dos listas de columnas:

- una lista de columnas numéricas,
- una lista de columnas categóricas.

Las variables numéricas serán medidas físicas del pingüino. Las variables categóricas serán variables como isla y sexo.

Estas listas se usarán para construir el preprocesador.

**Qué deberías comprobar al terminar esta celda**

Deberías tener una lista con columnas numéricas como `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` y `body_mass_g`, y una lista categórica con `island` y `sex`.


In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Columnas numéricas:")
print(numeric_features)

print("\nColumnas categóricas:")
print(categorical_features)

## Celda 9 — Separación en train y test

Divide `X` e `y` en entrenamiento y test.

Usa:

- 80% de datos para entrenamiento,
- 20% de datos para test,
- `random_state=42`,
- separación estratificada usando la variable objetivo.

La estratificación ayuda a mantener una proporción parecida de especies en train y test.

**Qué deberías comprobar al terminar esta celda**

Deberías obtener cuatro objetos: `X_train`, `X_test`, `y_train` e `y_test`, o nombres equivalentes. La distribución de especies debería ser parecida en entrenamiento y test.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nDistribución de clases en train:")
display(y_train.value_counts(normalize=True))

print("\nDistribución de clases en test:")
display(y_test.value_counts(normalize=True))

## Celda 10 — Construcción del preprocesador

Construye un preprocesador que haga:

- escalado estándar de las columnas numéricas,
- codificación one-hot de las columnas categóricas.

El objetivo es que los modelos reciban una matriz completamente numérica.

No ajustes el preprocesador manualmente con todos los datos. Lo recomendable es integrarlo dentro de cada pipeline para que se ajuste solo con los datos de entrenamiento.

**Qué deberías comprobar al terminar esta celda**

Al final de esta celda debes tener un objeto de preprocesado preparado para usarse dentro de pipelines.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocesador creado.")
print("- Variables numéricas: escalado con StandardScaler")
print("- Variables categóricas: codificación OneHotEncoder")

## Celda 11 — Baseline de clasificación

Entrena un clasificador baseline muy sencillo.

Por ejemplo, puedes usar un modelo que siempre prediga la clase más frecuente.

Después evalúalo sobre test.

Métricas mínimas:

- accuracy,
- F1 macro,
- matriz de confusión.

Guarda los resultados principales para construir una tabla comparativa al final.

**Qué deberías comprobar al terminar esta celda**

El baseline te dirá qué resultado se puede conseguir sin aprender patrones reales. Los modelos serios deberían superarlo claramente.


In [ ]:
results = []

def evaluate_classification_model(model_name, model, X_train, X_test, y_train, y_test):
    """Entrena un modelo de clasificación, calcula predicciones y devuelve métricas."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    results.append({
        "modelo": model_name,
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    })

    print(f"Modelo: {model_name}")
    print(f"Accuracy       : {accuracy:.4f}")
    print(f"Precision macro: {precision:.4f}")
    print(f"Recall macro   : {recall:.4f}")
    print(f"F1 macro       : {f1:.4f}")

    return model, y_pred

baseline = DummyClassifier(strategy="most_frequent")
baseline, y_pred_baseline = evaluate_classification_model(
    "Baseline - clase mayoritaria",
    baseline,
    X_train,
    X_test,
    y_train,
    y_test
)

## Celda 12 — Logistic Regression

Construye un pipeline con dos pasos:

1. el preprocesador,
2. un modelo de regresión logística para clasificación.

Aunque se llame “regresión logística”, en este contexto es un clasificador.

Entrena con los datos de entrenamiento, predice sobre test y calcula las métricas principales.

Guarda los resultados con el nombre del modelo.

**Qué deberías comprobar al terminar esta celda**

Deberías obtener un primer clasificador real y sus métricas sobre test.


In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

logistic_model, y_pred_logistic = evaluate_classification_model(
    "Logistic Regression",
    logistic_model,
    X_train,
    X_test,
    y_train,
    y_test
)

## Celda 13 — KNN Classifier

Construye un pipeline con el preprocesador y un clasificador KNN.

Elige un número inicial razonable de vecinos.

Entrena, predice y evalúa con las mismas métricas.

Recuerda que KNN es sensible a la escala de las variables, por lo que el escalado de columnas numéricas es importante.

**Qué deberías comprobar al terminar esta celda**

Deberías añadir una nueva fila de resultados comparables a los de la regresión logística.


In [ ]:
knn_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn_model, y_pred_knn = evaluate_classification_model(
    "KNN Classifier",
    knn_model,
    X_train,
    X_test,
    y_train,
    y_test
)

## Celda 14 — Support Vector Classifier

Construye un pipeline con el preprocesador y un Support Vector Classifier.

Puedes empezar con una configuración sencilla.

Entrena el modelo, predice sobre test y calcula las métricas.

Si quieres obtener probabilidades, recuerda que algunos clasificadores necesitan activar esa opción explícitamente. Para este ejercicio no es obligatorio trabajar con probabilidades.

**Qué deberías comprobar al terminar esta celda**

Deberías tener otra fila de resultados en la comparación de modelos.


In [ ]:
svc_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", SVC(kernel="rbf", C=1.0, probability=True, random_state=RANDOM_STATE))
])

svc_model, y_pred_svc = evaluate_classification_model(
    "Support Vector Classifier",
    svc_model,
    X_train,
    X_test,
    y_train,
    y_test
)

## Celda 15 — Decision Tree Classifier

Construye un pipeline con el preprocesador y un árbol de decisión para clasificación.

Entrena, predice y evalúa.

Puedes limitar la profundidad máxima para reducir el riesgo de sobreajuste. Si no la limitas, observa si el árbol se comporta de forma demasiado ajustada al entrenamiento.

**Qué deberías comprobar al terminar esta celda**

Deberías tener un modelo interpretable y comparable con los anteriores.


In [ ]:
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=4,
        random_state=RANDOM_STATE
    ))
])

tree_model, y_pred_tree = evaluate_classification_model(
    "Decision Tree Classifier",
    tree_model,
    X_train,
    X_test,
    y_train,
    y_test
)

## Celda 16 — Random Forest Classifier

Construye un pipeline con el preprocesador y un Random Forest de clasificación.

Usa un número razonable de árboles y fija `random_state=42`.

Entrena, predice y evalúa igual que con el resto de modelos.

**Qué deberías comprobar al terminar esta celda**

Deberías obtener la última fila de la comparación principal.


In [ ]:
forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

forest_model, y_pred_forest = evaluate_classification_model(
    "Random Forest Classifier",
    forest_model,
    X_train,
    X_test,
    y_train,
    y_test
)

## Celda 17 — Tabla comparativa de clasificadores

Construye una tabla final con todos los modelos evaluados.

La tabla debe tener, como mínimo:

- nombre del modelo,
- accuracy en test,
- F1 macro en test.

Puedes añadir también precision macro y recall macro si las has calculado.

Ordena la tabla con un criterio razonable, por ejemplo por mayor F1 macro.

**Qué deberías comprobar al terminar esta celda**

Deberías poder identificar claramente qué modelo ha funcionado mejor en test.


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="f1_macro", ascending=False).reset_index(drop=True)

display(results_df)

best_model_name = results_df.loc[0, "modelo"]
print("Mejor modelo según F1 macro:", best_model_name)

## Celda 18 — Matriz de confusión del mejor modelo

Elige el mejor modelo según tu criterio principal y calcula su matriz de confusión sobre el conjunto de test.

Visualiza la matriz de confusión de forma clara, con los nombres de las especies en los ejes.

Interpreta qué clases se confunden más entre sí.

**Qué deberías comprobar al terminar esta celda**

La matriz de confusión debe permitirte ver aciertos y errores por clase, no solo una métrica global.


In [ ]:
model_predictions = {
    "Baseline - clase mayoritaria": y_pred_baseline,
    "Logistic Regression": y_pred_logistic,
    "KNN Classifier": y_pred_knn,
    "Support Vector Classifier": y_pred_svc,
    "Decision Tree Classifier": y_pred_tree,
    "Random Forest Classifier": y_pred_forest
}

best_predictions = model_predictions[best_model_name]

cm = confusion_matrix(y_test, best_predictions, labels=sorted(y.unique()))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=sorted(y.unique())
)

disp.plot(values_format="d")
plt.title(f"Matriz de confusión - {best_model_name}")
plt.show()

## Celda 19 — Informe de clasificación

Para el mejor modelo, genera un informe de clasificación con métricas por clase.

Debes observar, como mínimo:

- precision,
- recall,
- F1-score,
- soporte de cada clase.

Después interpreta si el modelo funciona igual de bien para todas las especies o si alguna clase es más difícil.

**Qué deberías comprobar al terminar esta celda**

Deberías obtener una visión más detallada que con accuracy. Es posible que el modelo tenga mejor rendimiento en unas especies que en otras.


In [ ]:
print(f"Informe de clasificación del mejor modelo: {best_model_name}")
print()
print(classification_report(y_test, best_predictions, zero_division=0))

## Celda 20 — Discusión final

Escribe una conclusión breve, entre 8 y 12 líneas, respondiendo a estas preguntas:

1. ¿Qué modelo ha funcionado mejor según F1 macro?
2. ¿Ese modelo supera claramente al baseline?
3. ¿La accuracy cuenta toda la historia o hace falta mirar otras métricas?
4. ¿Qué especies se clasifican mejor?
5. ¿Qué especies se confunden más?
6. ¿Qué variables parecen más útiles para separar especies?
7. ¿Qué harías como siguiente mejora si tuvieras más tiempo?

Recuerda: la discusión debe conectar resultados, métricas y comprensión del problema.


In [ ]:
discussion = """
Discusión orientativa:

1. El baseline predice siempre la clase mayoritaria y sirve como referencia mínima.
2. En clasificación no basta con accuracy: si las clases están desbalanceadas, precision, recall y F1 ayudan a interpretar mejor el rendimiento.
3. Usamos F1 macro porque calcula la métrica por clase y luego promedia, dando importancia similar a todas las especies.
4. Las variables morfológicas de los pingüinos separan bastante bien las especies, especialmente medidas del pico, aleta y masa corporal.
5. Logistic Regression, SVC y KNN necesitan que las variables numéricas estén escaladas para funcionar correctamente.
6. Los árboles y Random Forest son más flexibles y capturan relaciones no lineales, aunque conviene vigilar el sobreajuste.
7. La matriz de confusión permite ver entre qué especies se equivoca el modelo, no solo cuántos aciertos tiene.
"""

print(discussion)

## Extra opcional — Validación cruzada o búsqueda de hiperparámetros

Si terminas pronto, elige **uno** de los modelos anteriores y prueba una pequeña mejora.

Opciones posibles:

- usar validación cruzada para comparar modelos de forma más robusta,
- ajustar el número de vecinos en KNN,
- ajustar el parámetro `C` o el kernel en SVC,
- ajustar la profundidad máxima de un árbol,
- ajustar algunos hiperparámetros de Random Forest.

Después compara el resultado original con el resultado ajustado. No basta con decir que mejora: debes mostrarlo con métricas.


In [ ]:
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"]
}

grid_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier())
])

grid_search = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Mejores hiperparámetros:")
print(grid_search.best_params_)

best_grid_model = grid_search.best_estimator_
y_pred_grid = best_grid_model.predict(X_test)

print("\nEvaluación del modelo optimizado:")
print("Accuracy:", accuracy_score(y_test, y_pred_grid))
print("F1 macro:", f1_score(y_test, y_pred_grid, average="macro"))
print("\nInforme completo:")
print(classification_report(y_test, y_pred_grid, zero_division=0))

## Checklist antes de entregar

Antes de entregar, revisa que tu notebook cumple lo siguiente:

- el notebook se ejecuta de arriba abajo sin errores,
- el dataset usado es el indicado en este ejercicio,
- existe una separación clara entre `X` e `y`,
- hay train/test split estratificado,
- las variables categóricas se transforman correctamente,
- las variables numéricas se escalan cuando corresponde,
- se comparan al menos cinco clasificadores,
- se usan accuracy y F1 macro,
- hay matriz de confusión del mejor modelo,
- hay informe de clasificación del mejor modelo,
- hay una conclusión escrita con criterio.
